# Hourly Delivery Demand Forecasting — Training Pipeline

**Objective:** Train and rigorously evaluate models to predict `demand_count` per `(city, region_id, aoi_id, hour)` using the advanced feature set from the feature engineering notebook (~59 features across 10 categories).

**Pipeline design principles:**
1. **Ablation-first** — Every feature group must prove its value beyond lag-only baselines.
2. **No leakage** — Target encoding fitted on train only; chronological splits; no future data in features.
3. **Reproducibility** — Single config section; every experiment logged with full parameter set.
4. **Temporal stability** — Walk-forward validation, not just one test score.
5. **Diagnostic depth** — Residual analysis, overfitting checks, permutation importance.
6. **Fair comparison** — Multiple AOI profiles (busy/medium/sparse), not just cherry-picked examples.

## 1. Configuration & Setup

All hyperparameters, paths, and toggles live here. Nothing else in the notebook should be edited for a new experiment run.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import warnings
import os
import json
import gc
from datetime import datetime
from collections import OrderedDict

warnings.filterwarnings('ignore')

# ── Install dependencies if needed ────────────────────────────────────────────
for pkg, name in [("xgboost", "xgb"), ("lightgbm", "lgb"), ("sklearn", "sklearn")]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run(["pip", "install", pkg, "-q"])

import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error

print(f"XGBoost {xgb.__version__}, LightGBM {lgb.__version__}")

# ══════════════════════════════════════════════════════════════════════════════
# MASTER CONFIGURATION — edit ONLY this block for new experiments
# ══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Paths ──
    "csv_path":        "lade_hourly_features.csv",  # local path; change for Colab
    "model_dir":       "models/",
    "experiment_log":  "experiments.json",

    # ── Reproducibility ──
    "seed": 42,

    # ── Split ratios (chronological) ──
    "train_ratio": 0.60,
    "val_ratio":   0.20,
    # test = 1 - train - val = 0.20

    # ── Walk-forward validation ──
    "wf_n_splits": 3,          # number of expanding-window folds

    # ── Training caps ──
    "max_train_rows": None,     # None = use all; set int to cap

    # ── XGBoost grid (structured variation) ──
    "xgb_configs": [
        {
            "name": "XGB-Poisson-Base",
            "objective": "count:poisson",
            "n_estimators": 3000,
            "learning_rate": 0.05,
            "max_depth": 6,
            "min_child_weight": 10,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
            "reg_lambda": 2.0,
        },
        {
            "name": "XGB-Poisson-Deep",
            "objective": "count:poisson",
            "n_estimators": 3000,
            "learning_rate": 0.03,
            "max_depth": 8,
            "min_child_weight": 15,
            "subsample": 0.7,
            "colsample_bytree": 0.7,
            "reg_alpha": 0.5,
            "reg_lambda": 5.0,
        },
        {
            "name": "XGB-Poisson-Shallow",
            "objective": "count:poisson",
            "n_estimators": 3000,
            "learning_rate": 0.1,
            "max_depth": 4,
            "min_child_weight": 5,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
        },
        {
            "name": "XGB-LogError",
            "objective": "reg:squaredlogerror",
            "n_estimators": 3000,
            "learning_rate": 0.05,
            "max_depth": 6,
            "min_child_weight": 10,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
            "reg_lambda": 2.0,
        },
    ],

    # ── LightGBM config ──
    "lgb_config": {
        "name": "LightGBM-Poisson",
        "objective": "poisson",
        "n_estimators": 3000,
        "learning_rate": 0.05,
        "max_depth": 6,  # -1 for unlimited
        "num_leaves": 63,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 2.0,
    },

    # ── Ridge (linear sanity check) ──
    "ridge_alpha": 1.0,

    # ── Early stopping ──
    "early_stopping_rounds": 100,
}

SEED = CONFIG["seed"]
np.random.seed(SEED)

# ── GPU detection ─────────────────────────────────────────────────────────────
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
except ImportError:
    GPU_AVAILABLE = False

TREE_METHOD = "gpu_hist" if GPU_AVAILABLE else "hist"
print(f"Tree method: {TREE_METHOD} (GPU: {GPU_AVAILABLE})")
print(f"Config loaded: {len(CONFIG['xgb_configs'])} XGB variants + LightGBM + Ridge")

## 2. Feature Group Definitions

Explicit categorization enables ablation experiments. Each group is a list of column names.

**Why this matters:** The old pipeline treated all features as a flat list. By grouping them, we can
test whether temporal context, spatial features, and interaction terms add value beyond raw lags —
the core question this pipeline must answer.

In [ ]:
# ── Feature Groups (must match feature engineering output) ─────────────────────
# These are the column groupings from the feature engineering notebook.

FEATURE_GROUPS = OrderedDict({
    "calendar":    ["hour", "dow", "month", "is_weekend", "day_of_month", "week_of_year"],
    "cyclical":    ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"],
    "peak":        ["is_peak_hour"],
    "interaction": ["hour_sin_x_weekend", "hour_cos_x_weekend", "is_peak_x_weekend"],
    "spatial":     ["aoi_lng", "aoi_lat", "aoi_busyness"],
    "lag":         ["lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "lag_24", "lag_48", "lag_168"],
    "rolling":     [f"roll_{w}_{s}" for w in [6, 24, 72, 168]
                    for s in ["mean", "std", "min", "max", "range"]],
    "momentum":    ["momentum_1_24", "acceleration", "lag1_over_roll24",
                    "lag1_over_roll168", "trend_24_168", "trend_6_24"],
    "zero_demand": ["is_nonzero", "zero_ratio_24", "consec_zeros"],
    "neighbor":    ["region_lag1_mean", "region_lag24_mean", "aoi_vs_region"],
})

# Derived feature sets for ablation
LAG_ONLY_FEATURES   = FEATURE_GROUPS["lag"]
NO_LAG_FEATURES     = [c for g, cols in FEATURE_GROUPS.items()
                       if g not in ("lag", "rolling", "momentum") for c in cols]
ALL_FEATURES        = [c for cols in FEATURE_GROUPS.values() for c in cols]

# Columns added during preprocessing (target encoding, one-hot)
EXTRA_FEATURES = []  # populated after encoding

ID_COLS   = ["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"]
TARGET    = "demand_count"
DROP_COLS = ID_COLS + [TARGET, "split"]

print(f"Feature groups: {len(FEATURE_GROUPS)}")
for name, cols in FEATURE_GROUPS.items():
    print(f"  {name:12s}: {len(cols)} features")
print(f"\nTotal raw features: {len(ALL_FEATURES)}")
print(f"  Lag-only:  {len(LAG_ONLY_FEATURES)}")
print(f"  No-lag:    {len(NO_LAG_FEATURES)}")

## 3. Data Loading & Memory Optimization

Load the preprocessed CSV once and downcast types to minimize memory.

In [ ]:
# ── Mount Google Drive if running in Colab ─────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # Uncomment and adjust if your CSV is on Drive:
    # CONFIG["csv_path"] = "/content/drive/MyDrive/Senior Project/LaDe-Dataset/lade_hourly_features.csv"
except ImportError:
    pass  # Not in Colab

# ── Load ──────────────────────────────────────────────────────────────────────
print(f"Loading {CONFIG['csv_path']}...")
df = pd.read_csv(CONFIG["csv_path"], parse_dates=["bucket_hour"])
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")

# ── Type optimization ─────────────────────────────────────────────────────────
int_cols = ["region_id", "aoi_id", "hour", "dow", "month", "is_weekend",
            "day_of_month", "week_of_year", "is_peak_hour", "is_peak_x_weekend",
            "is_nonzero", "consec_zeros"]
for c in int_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")

float_cols = [c for c in df.columns if df[c].dtype == "float64"]
for c in float_cols:
    df[c] = df[c].astype("float32")

df[TARGET] = df[TARGET].astype("int32")
mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"Memory: {mem_gb:.2f} GB")

# ── Validate expected columns exist ──────────────────────────────────────────
missing = [c for c in ALL_FEATURES if c not in df.columns]
if missing:
    print(f"WARNING: Missing features: {missing}")
else:
    print(f"All {len(ALL_FEATURES)} expected features present.")

df.head(3)

## 4. Chronological Train / Val / Test Split

**Why chronological:** Time-series data has temporal autocorrelation. Random splits leak future
information into training. We split strictly by time so the model never sees future demand patterns.

- **Train:** first 60% of unique hours
- **Val:** next 20%
- **Test:** last 20%

In [ ]:
all_hours = np.sort(df["bucket_hour"].unique())
n_hours   = len(all_hours)

train_end = all_hours[int(n_hours * CONFIG["train_ratio"]) - 1]
val_end   = all_hours[int(n_hours * (CONFIG["train_ratio"] + CONFIG["val_ratio"])) - 1]

df["split"] = np.where(
    df["bucket_hour"] <= train_end, "train",
    np.where(df["bucket_hour"] <= val_end, "val", "test")
)

for s in ["train", "val", "test"]:
    n = (df["split"] == s).sum()
    hours = df.loc[df["split"] == s, "bucket_hour"]
    print(f"{s:5s}: {n:>10,} rows | {hours.min()} → {hours.max()}")

print(f"\nSplit boundaries: train_end={train_end}, val_end={val_end}")

## 5. Target Encoding — Strict Train-Only Fit

**Problem:** `aoi_id` has too many levels for one-hot encoding, but using it as a raw integer
encodes an arbitrary ordering. Target encoding maps each AOI to its mean demand, providing a
meaningful numeric representation.

**Leakage prevention:**
- `TargetEncoder.fit()` is called **only on training rows**.
- Val and test are transformed using the train-fitted encoder.
- This is equivalent to out-of-fold encoding restricted to the train set.
- The encoder uses sklearn's built-in internal CV with smoothing to avoid overfitting to
  individual AOI means even within train.

**Why not fit_transform on full data:** The old notebook called fit_transform on the entire
dataframe. Even though sklearn's TargetEncoder uses internal CV, fitting on val/test rows
allows the encoder to see future demand distributions, which is a subtle form of leakage.

In [ ]:
train_mask = df["split"] == "train"

# ── Target encode aoi_id: fit on train, transform all ────────────────────────
te = TargetEncoder(target_type="continuous", smooth="auto", random_state=SEED)
te.fit(df.loc[train_mask, ["aoi_id"]].astype("int64"),
       df.loc[train_mask, TARGET].astype("float64"))

df["aoi_id_encoded"] = te.transform(df[["aoi_id"]].astype("int64")).astype("float32")

# ── One-hot encode low-cardinality categoricals ──────────────────────────────
OHE_COLS = ["city", "aoi_type"]
df_ohe = pd.get_dummies(df[OHE_COLS], prefix=OHE_COLS, dtype="int8")

# Track extra feature columns
EXTRA_FEATURES.extend(["aoi_id_encoded"] + list(df_ohe.columns))
FULL_FEATURE_COLS = ALL_FEATURES + EXTRA_FEATURES

print(f"Target-encoded aoi_id (train-only fit). Encoded range: "
      f"{df['aoi_id_encoded'].min():.2f} — {df['aoi_id_encoded'].max():.2f}")
print(f"One-hot columns: {list(df_ohe.columns)}")
print(f"Total feature count: {len(FULL_FEATURE_COLS)}")

# ── Build encoded feature matrix ─────────────────────────────────────────────
encoded = pd.concat([df[ALL_FEATURES + ["aoi_id_encoded"]], df_ohe], axis=1).astype("float32")

# Verify no target leakage
assert TARGET not in encoded.columns, "Target column in feature matrix!"
assert "bucket_hour" not in encoded.columns, "Timestamp in feature matrix!"
print("Leakage check passed.")

## 6. Prepare Train / Val / Test Arrays

In [ ]:
split_mask = df["split"]

train_idx = df.index[split_mask == "train"]
val_idx   = df.index[split_mask == "val"]
test_idx  = df.index[split_mask == "test"]

# Optional training cap
if CONFIG["max_train_rows"] and len(train_idx) > CONFIG["max_train_rows"]:
    rng = np.random.RandomState(SEED)
    train_idx = train_idx[rng.choice(len(train_idx), CONFIG["max_train_rows"], replace=False)]
    print(f"Train capped to {CONFIG['max_train_rows']:,} rows")

X_train = encoded.loc[train_idx].astype("float32")
X_val   = encoded.loc[val_idx].astype("float32")
X_test  = encoded.loc[test_idx].astype("float32")

y_train = df.loc[train_idx, TARGET].values.astype("float32")
y_val   = df.loc[val_idx, TARGET].values.astype("float32")
y_test  = df.loc[test_idx, TARGET].values.astype("float32")

# Store metadata for later visualization (before freeing df)
test_meta = df.loc[test_idx, ["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"]].copy()
val_meta  = df.loc[val_idx, ["city", "region_id", "aoi_id", "aoi_type", "bucket_hour"]].copy()

print(f"X_train: {X_train.shape}  ({X_train.memory_usage(deep=True).sum()/1e9:.2f} GB)")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y range: train [{y_train.min():.0f}, {y_train.max():.0f}], "
      f"val [{y_val.min():.0f}, {y_val.max():.0f}], "
      f"test [{y_test.min():.0f}, {y_test.max():.0f}]")

assert (y_train >= 0).all() and (y_val >= 0).all() and (y_test >= 0).all(), \
    "Negative demand values detected — Poisson objective will fail"

# ── Free main dataframe ──────────────────────────────────────────────────────
del df, df_ohe
gc.collect()
print("Freed raw dataframe.")

## 7. Evaluation Metrics & Experiment Logger

Centralized metric computation and experiment tracking. Every model run is logged with its
full parameter set so results are reproducible and comparable.

In [ ]:
def compute_metrics(y_true, y_pred, prefix=""):
    """Compute all evaluation metrics. Returns a dict."""
    y_pred_clipped = np.clip(y_pred, 0, None)
    y_true = np.asarray(y_true, dtype="float64")
    y_pred_clipped = np.asarray(y_pred_clipped, dtype="float64")

    mae_val = np.mean(np.abs(y_true - y_pred_clipped))
    rmse_val = np.sqrt(np.mean((y_true - y_pred_clipped) ** 2))

    # RMSLE — robust to scale differences
    rmsle_val = np.sqrt(np.mean(
        (np.log1p(y_true) - np.log1p(y_pred_clipped)) ** 2
    ))

    # Poisson deviance — natural metric for count data
    yp = np.clip(y_pred_clipped, 1e-8, None)
    poisson_dev = 2 * np.mean(
        np.where(y_true == 0, yp, y_true * np.log(y_true / yp) - y_true + yp)
    )

    # Median absolute error — robust to outliers
    medae = np.median(np.abs(y_true - y_pred_clipped))

    p = f"{prefix}_" if prefix else ""
    return {
        f"{p}mae": round(mae_val, 4),
        f"{p}rmse": round(rmse_val, 4),
        f"{p}rmsle": round(rmsle_val, 4),
        f"{p}poisson_dev": round(poisson_dev, 4),
        f"{p}medae": round(medae, 4),
    }


# ── Experiment Logger ─────────────────────────────────────────────────────────
EXPERIMENT_LOG = []

def log_experiment(name, params, val_metrics, test_metrics, extra=None):
    """Log a complete experiment run."""
    record = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "name": name,
        "params": params,
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }
    if extra:
        record.update(extra)
    EXPERIMENT_LOG.append(record)
    print(f"  Logged: {name} | val_mae={val_metrics.get('mae', 'N/A')} | "
          f"test_mae={test_metrics.get('mae', 'N/A')}")
    return record

def show_experiment_summary():
    """Display all logged experiments as a comparison table."""
    if not EXPERIMENT_LOG:
        print("No experiments logged yet.")
        return pd.DataFrame()
    rows = []
    for exp in EXPERIMENT_LOG:
        rows.append({
            "Model": exp["name"],
            "Val MAE": exp.get("val_mae"),
            "Val RMSE": exp.get("val_rmse"),
            "Val RMSLE": exp.get("val_rmsle"),
            "Test MAE": exp.get("test_mae"),
            "Test RMSE": exp.get("test_rmse"),
            "Test RMSLE": exp.get("test_rmsle"),
            "Test Poisson": exp.get("test_poisson_dev"),
        })
    summary = pd.DataFrame(rows).sort_values("Test MAE")
    return summary

print("Metrics and logging ready.")

## 8. Baselines — Lag-24 and Rolling Mean

**Why these baselines matter:** If a complex model cannot beat "just use yesterday's value at the
same hour" or "average of the last 24 hours", the model adds no value. These baselines also
establish the improvement margin: a model that beats lag-24 by only 1% is not worth the
complexity, while 15%+ improvement justifies the engineering effort.

In [ ]:
# ── Baseline: Lag-24 (same hour yesterday) ────────────────────────────────────
if "lag_24" in X_val.columns:
    baseline_lag24_val  = X_val["lag_24"].values
    baseline_lag24_test = X_test["lag_24"].values

    val_m  = compute_metrics(y_val, baseline_lag24_val)
    test_m = compute_metrics(y_test, baseline_lag24_test)
    log_experiment("Baseline: Lag-24", {"method": "lag_24"}, val_m, test_m)

    print(f"Lag-24 Baseline — Val MAE: {val_m['mae']:.4f}, Test MAE: {test_m['mae']:.4f}")
else:
    print("WARNING: lag_24 not in features — cannot compute lag baseline")

# ── Baseline: Rolling-24 mean ─────────────────────────────────────────────────
if "roll_24_mean" in X_val.columns:
    baseline_roll_val  = X_val["roll_24_mean"].values
    baseline_roll_test = X_test["roll_24_mean"].values

    val_m  = compute_metrics(y_val, baseline_roll_val)
    test_m = compute_metrics(y_test, baseline_roll_test)
    log_experiment("Baseline: Roll-24-Mean", {"method": "roll_24_mean"}, val_m, test_m)

    print(f"Roll-24 Baseline — Val MAE: {val_m['mae']:.4f}, Test MAE: {test_m['mae']:.4f}")

# ── Baseline: Global mean (trivial) ──────────────────────────────────────────
global_mean = y_train.mean()
val_m  = compute_metrics(y_val, np.full_like(y_val, global_mean))
test_m = compute_metrics(y_test, np.full_like(y_test, global_mean))
log_experiment("Baseline: Global Mean", {"method": "global_mean", "value": float(global_mean)}, val_m, test_m)
print(f"Global Mean Baseline ({global_mean:.2f}) — Val MAE: {val_m['mae']:.4f}, Test MAE: {test_m['mae']:.4f}")

## 9. Feature Ablation — Does the New Feature Set Matter?

**The core experiment:** The old pipeline was lag-dominated. We test three feature sets:
1. **Lag-only:** Just the 8 lag features — pure autoregressive baseline.
2. **No-lag:** All engineered features EXCEPT lags/rolling/momentum — tests whether temporal
   context, spatial, and interaction features can predict demand without historical values.
3. **Full features:** Everything — tests whether engineered features add value ON TOP of lags.

All three use the same model (XGBoost Poisson, base config) for fair comparison.

**Why this answers the right question:** If Full ≈ Lag-only, the engineering effort was wasted.
If Full >> Lag-only, the new features genuinely help. If No-lag is competitive, the model can
potentially generalize to cold-start AOIs with no history.

In [ ]:
def train_xgb_quick(X_tr, y_tr, X_v, y_v, name, objective="count:poisson"):
    """Train a single XGBoost model with base config for ablation comparison."""
    model = xgb.XGBRegressor(
        objective=objective,
        tree_method=TREE_METHOD,
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=2.0,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_v, y_v)],
        verbose=False,
    )
    return model

# ── Ablation feature sets ─────────────────────────────────────────────────────
ablation_sets = {
    "Lag-Only": [c for c in LAG_ONLY_FEATURES if c in encoded.columns],
    "No-Lag":   [c for c in NO_LAG_FEATURES + EXTRA_FEATURES if c in encoded.columns],
    "Full":     [c for c in FULL_FEATURE_COLS if c in encoded.columns],
}

ablation_results = {}
for set_name, feature_cols in ablation_sets.items():
    print(f"\nTraining ablation: {set_name} ({len(feature_cols)} features)...")

    mdl = train_xgb_quick(
        X_train[feature_cols], y_train,
        X_val[feature_cols], y_val,
        set_name
    )

    val_pred  = np.clip(mdl.predict(X_val[feature_cols]), 0, None)
    test_pred = np.clip(mdl.predict(X_test[feature_cols]), 0, None)

    val_m  = compute_metrics(y_val, val_pred)
    test_m = compute_metrics(y_test, test_pred)
    log_experiment(f"Ablation: {set_name}", {"features": set_name, "n_features": len(feature_cols)}, val_m, test_m)
    ablation_results[set_name] = {"val": val_m, "test": test_m, "model": mdl}

    print(f"  {set_name}: Val MAE={val_m['mae']:.4f}, Test MAE={test_m['mae']:.4f}, "
          f"Test RMSLE={test_m['rmsle']:.4f}")

# ── Ablation comparison ──────────────────────────────────────────────────────
print("\n" + "="*70)
print("ABLATION RESULTS — Feature Set Comparison")
print("="*70)
lag_only_mae = ablation_results["Lag-Only"]["test"]["mae"]
for name, res in ablation_results.items():
    t = res["test"]
    improvement = (1 - t["mae"] / lag_only_mae) * 100 if lag_only_mae > 0 else 0
    print(f"  {name:10s}: MAE={t['mae']:.4f}  RMSE={t['rmse']:.4f}  "
          f"RMSLE={t['rmsle']:.4f}  vs Lag-Only: {improvement:+.1f}%")

## 10. XGBoost — Structured Hyperparameter Exploration

**Why structured variation instead of random/grid search:** With 4 configs that vary depth,
learning rate, and regularization in meaningful combinations, we cover the bias-variance
tradeoff efficiently. Each config tests a hypothesis:

- **Base:** Balanced tradeoff — the workhorse config.
- **Deep:** Higher capacity, stronger regularization — for complex patterns.
- **Shallow:** Lower capacity, less regularization — for simpler/smoother patterns.
- **LogError:** Different loss function — tests if squared-log-error handles the heavy tail better.

In [ ]:
FEATURE_COLS = [c for c in FULL_FEATURE_COLS if c in encoded.columns]
xgb_models = {}
xgb_evals  = {}

for cfg in CONFIG["xgb_configs"]:
    name = cfg["name"]
    print(f"\nTraining {name}...")

    params = {k: v for k, v in cfg.items() if k != "name"}
    params["tree_method"] = TREE_METHOD
    params["random_state"] = SEED
    params["n_jobs"] = -1

    model = xgb.XGBRegressor(**params)
    model.fit(
        X_train[FEATURE_COLS], y_train,
        eval_set=[(X_train[FEATURE_COLS], y_train), (X_val[FEATURE_COLS], y_val)],
        verbose=False,
    )

    val_pred  = np.clip(model.predict(X_val[FEATURE_COLS]), 0, None)
    test_pred = np.clip(model.predict(X_test[FEATURE_COLS]), 0, None)

    val_m  = compute_metrics(y_val, val_pred)
    test_m = compute_metrics(y_test, test_pred)

    extra = {"best_iteration": model.best_iteration if hasattr(model, 'best_iteration') else None}
    log_experiment(name, params, val_m, test_m, extra)

    xgb_models[name] = model
    xgb_evals[name]  = model.evals_result()

    print(f"  Val MAE: {val_m['mae']:.4f} | Test MAE: {test_m['mae']:.4f} | "
          f"Test RMSLE: {test_m['rmsle']:.4f} | "
          f"Best iter: {extra['best_iteration']}")

# Identify best XGBoost
best_xgb_name = min(xgb_models, key=lambda n: compute_metrics(y_val, np.clip(xgb_models[n].predict(X_val[FEATURE_COLS]), 0, None))["mae"])
best_xgb = xgb_models[best_xgb_name]
print(f"\nBest XGBoost config: {best_xgb_name}")

## 11. LightGBM — Alternative Gradient Boosting

**Why include LightGBM:** It uses histogram-based splitting with a different leaf-wise growth
strategy. If it significantly outperforms XGBoost, it suggests the data has patterns better
captured by deeper, narrower trees. If XGBoost wins, the data favors balanced trees.
This comparison sanity-checks that our results aren't an artifact of one library's defaults.

In [ ]:
cfg = CONFIG["lgb_config"]
lgb_name = cfg["name"]
print(f"Training {lgb_name}...")

lgb_params = {k: v for k, v in cfg.items() if k != "name"}
lgb_params["random_state"] = SEED
lgb_params["n_jobs"] = -1
lgb_params["verbose"] = -1

lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(
    X_train[FEATURE_COLS], y_train,
    eval_set=[(X_val[FEATURE_COLS], y_val)],
    eval_metric="mae",
    callbacks=[
        lgb.early_stopping(CONFIG["early_stopping_rounds"], verbose=False),
        lgb.log_evaluation(period=0),
    ],
)

val_pred  = np.clip(lgb_model.predict(X_val[FEATURE_COLS]), 0, None)
test_pred = np.clip(lgb_model.predict(X_test[FEATURE_COLS]), 0, None)

val_m  = compute_metrics(y_val, val_pred)
test_m = compute_metrics(y_test, test_pred)
log_experiment(lgb_name, lgb_params, val_m, test_m,
               {"best_iteration": lgb_model.best_iteration_})

print(f"  Val MAE: {val_m['mae']:.4f} | Test MAE: {test_m['mae']:.4f} | "
      f"Test RMSLE: {test_m['rmsle']:.4f} | Best iter: {lgb_model.best_iteration_}")

## 12. Ridge Regression — Linear Sanity Check

**Why a linear model:** If Ridge achieves >80% of XGBoost's performance, most of the signal is
linear and the tree ensemble's complexity may not be justified. If XGBoost dramatically outperforms
Ridge, non-linear interactions (which trees capture naturally) are important for this data.

In [ ]:
# Ridge needs standardized features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[FEATURE_COLS])
X_val_scaled   = scaler.transform(X_val[FEATURE_COLS])
X_test_scaled  = scaler.transform(X_test[FEATURE_COLS])

ridge = Ridge(alpha=CONFIG["ridge_alpha"], random_state=SEED)
ridge.fit(X_train_scaled, y_train)

val_pred  = np.clip(ridge.predict(X_val_scaled), 0, None)
test_pred = np.clip(ridge.predict(X_test_scaled), 0, None)

val_m  = compute_metrics(y_val, val_pred)
test_m = compute_metrics(y_test, test_pred)
log_experiment("Ridge (Linear)", {"alpha": CONFIG["ridge_alpha"]}, val_m, test_m)

print(f"Ridge — Val MAE: {val_m['mae']:.4f} | Test MAE: {test_m['mae']:.4f} | "
      f"Test RMSLE: {test_m['rmsle']:.4f}")

del X_train_scaled, X_val_scaled, X_test_scaled
gc.collect()

## 13. Sample Weighting Experiment

**Old approach:** Weighted all samples by AOI busyness, giving busier zones more influence.

**Problem:** This biases the model toward high-demand zones and can hurt sparse-zone predictions.
For a demand forecasting system, accurate predictions in sparse zones may be equally important
(e.g., for courier allocation to underserved areas).

**Experiment:** Train the best XGB config with and without weights, compare performance across
AOI busyness tiers. Only use weights if they improve BOTH busy and sparse zone performance.

In [ ]:
# ── Compute sample weights (proportional to AOI mean demand) ──────────────────
if "aoi_busyness" in X_train.columns:
    # Normalize weights to mean=1 so loss scale doesn't change
    w_train = X_train["aoi_busyness"].values.copy()
    w_train = w_train / w_train.mean()
    w_train = np.clip(w_train, 0.1, 10.0)  # prevent extreme weights

    # ── Train best XGB config WITH weights ────────────────────────────────────
    best_cfg = [c for c in CONFIG["xgb_configs"] if c["name"] == best_xgb_name][0]
    params_w = {k: v for k, v in best_cfg.items() if k != "name"}
    params_w["tree_method"] = TREE_METHOD
    params_w["random_state"] = SEED
    params_w["n_jobs"] = -1

    print(f"Training {best_xgb_name} WITH sample weights...")
    model_weighted = xgb.XGBRegressor(**params_w)
    model_weighted.fit(
        X_train[FEATURE_COLS], y_train,
        sample_weight=w_train,
        eval_set=[(X_val[FEATURE_COLS], y_val)],
        verbose=False,
    )

    val_pred_w  = np.clip(model_weighted.predict(X_val[FEATURE_COLS]), 0, None)
    test_pred_w = np.clip(model_weighted.predict(X_test[FEATURE_COLS]), 0, None)

    val_mw  = compute_metrics(y_val, val_pred_w)
    test_mw = compute_metrics(y_test, test_pred_w)
    log_experiment(f"{best_xgb_name} (Weighted)", {**params_w, "weighted": True}, val_mw, test_mw)

    # ── Compare weighted vs unweighted by AOI busyness tier ───────────────────
    busyness = X_test["aoi_busyness"].values
    tiers = pd.qcut(busyness, q=3, labels=["sparse", "medium", "busy"])

    pred_unweighted = np.clip(best_xgb.predict(X_test[FEATURE_COLS]), 0, None)

    print(f"\n{'Tier':<10} {'Unweighted MAE':>16} {'Weighted MAE':>16} {'Better':>10}")
    print("-" * 55)
    weight_helps = True
    for tier in ["sparse", "medium", "busy"]:
        mask = tiers == tier
        mae_uw = np.mean(np.abs(y_test[mask] - pred_unweighted[mask]))
        mae_w  = np.mean(np.abs(y_test[mask] - test_pred_w[mask]))
        better = "weighted" if mae_w < mae_uw else "unweighted"
        if tier == "sparse" and mae_w > mae_uw:
            weight_helps = False
        print(f"{tier:<10} {mae_uw:>16.4f} {mae_w:>16.4f} {better:>10}")

    print(f"\nRecommendation: {'USE' if weight_helps else 'SKIP'} sample weights")
else:
    print("aoi_busyness not in features — skipping weight experiment")
    weight_helps = False

## 14. Walk-Forward Temporal Validation

**Why not just one val score:** A single train/val/test split tells us model performance at one
point in time. Walk-forward validation simulates real deployment: train on history, predict the
next window, expand history, repeat. This tests whether performance is **stable over time** or
whether it degrades as patterns shift.

**Implementation:** We split the combined train+val period into expanding windows:
- Fold 1: Train on first 40%, validate on next 20%
- Fold 2: Train on first 60%, validate on next 20%
- Fold 3: Train on first 80%, validate on last 20%

The standard deviation across folds reveals model stability.

In [ ]:
# ── Walk-Forward Validation ────────────────────────────────────────────────────
# We reconstruct df temporarily for splitting — only the columns we need
# Use the stored encoded features + target

all_data_X = pd.concat([X_train, X_val, X_test], axis=0)
all_data_y = np.concatenate([y_train, y_val, y_test])
all_data_hours = pd.concat([
    val_meta[["bucket_hour"]].reindex(X_train.index, fill_value=pd.NaT),
    val_meta[["bucket_hour"]],
    test_meta[["bucket_hour"]]
], axis=0)

# Simpler approach: use the original indices to get hours
# We know train comes first temporally, then val, then test
n_total = len(all_data_y)
n_splits = CONFIG["wf_n_splits"]

wf_results = []
print(f"Walk-Forward Validation ({n_splits} expanding folds)\n")

# Create fold boundaries as fractions of total data
for fold_i in range(n_splits):
    # Expanding train window
    train_frac = (fold_i + 1) / (n_splits + 1)
    val_start_frac = train_frac
    val_end_frac = train_frac + 1 / (n_splits + 1)

    tr_end = int(n_total * train_frac)
    vl_start = tr_end
    vl_end = min(int(n_total * val_end_frac), n_total)

    X_wf_train = all_data_X.iloc[:tr_end]
    y_wf_train = all_data_y[:tr_end]
    X_wf_val   = all_data_X.iloc[vl_start:vl_end]
    y_wf_val   = all_data_y[vl_start:vl_end]

    # Train with best config
    best_cfg = [c for c in CONFIG["xgb_configs"] if c["name"] == best_xgb_name][0]
    params_wf = {k: v for k, v in best_cfg.items() if k != "name"}
    params_wf["tree_method"] = TREE_METHOD
    params_wf["random_state"] = SEED
    params_wf["n_jobs"] = -1

    model_wf = xgb.XGBRegressor(**params_wf)
    model_wf.fit(X_wf_train[FEATURE_COLS], y_wf_train,
                 eval_set=[(X_wf_val[FEATURE_COLS], y_wf_val)],
                 verbose=False)

    pred_wf = np.clip(model_wf.predict(X_wf_val[FEATURE_COLS]), 0, None)
    fold_metrics = compute_metrics(y_wf_val, pred_wf)
    fold_metrics["fold"] = fold_i + 1
    fold_metrics["train_size"] = len(y_wf_train)
    fold_metrics["val_size"] = len(y_wf_val)
    wf_results.append(fold_metrics)

    print(f"  Fold {fold_i+1}: train={len(y_wf_train):,}, val={len(y_wf_val):,} | "
          f"MAE={fold_metrics['mae']:.4f}  RMSE={fold_metrics['rmse']:.4f}")

    del model_wf, X_wf_train, X_wf_val
    gc.collect()

# ── Stability report ─────────────────────────────────────────────────────────
wf_df = pd.DataFrame(wf_results)
print(f"\nWalk-Forward Stability:")
print(f"  MAE  — mean: {wf_df['mae'].mean():.4f}, std: {wf_df['mae'].std():.4f}, "
      f"range: [{wf_df['mae'].min():.4f}, {wf_df['mae'].max():.4f}]")
print(f"  RMSE — mean: {wf_df['rmse'].mean():.4f}, std: {wf_df['rmse'].std():.4f}")

cv_coeff = wf_df['mae'].std() / wf_df['mae'].mean() * 100
print(f"  Coefficient of variation: {cv_coeff:.1f}%")
if cv_coeff < 5:
    print("  Interpretation: Very stable across time periods.")
elif cv_coeff < 15:
    print("  Interpretation: Moderately stable — some temporal drift.")
else:
    print("  Interpretation: Unstable — model performance varies significantly over time.")

del all_data_X, all_data_y
gc.collect()

## 15. Full Results Comparison

All models and baselines ranked by test MAE with explicit improvement margins over baselines.

In [ ]:
summary = show_experiment_summary()
display(summary)

# ── Improvement margins over baselines ────────────────────────────────────────
baseline_lag24_mae = None
for exp in EXPERIMENT_LOG:
    if exp["name"] == "Baseline: Lag-24":
        baseline_lag24_mae = exp["test_mae"]
        break

if baseline_lag24_mae:
    print(f"\nImprovement over Lag-24 baseline (Test MAE = {baseline_lag24_mae:.4f}):")
    for exp in sorted(EXPERIMENT_LOG, key=lambda x: x.get("test_mae", 999)):
        if "Baseline" not in exp["name"]:
            imp = (1 - exp["test_mae"] / baseline_lag24_mae) * 100
            print(f"  {exp['name']:30s}: {exp['test_mae']:.4f}  ({imp:+.1f}%)")

# ── Save experiment log to disk ───────────────────────────────────────────────
log_path = CONFIG["experiment_log"]
with open(log_path, "w") as f:
    json.dump(EXPERIMENT_LOG, f, indent=2, default=str)
print(f"\nExperiment log saved to {log_path}")

## 16. Training Curves & Overfitting Diagnostics

**Beyond just plotting:** We automatically detect the train-val gap at convergence. A large gap
signals overfitting — the model memorizes training patterns that don't generalize.

**Interpretation guide:**
- Gap < 10% of val metric: healthy generalization
- Gap 10-25%: mild overfitting — consider more regularization
- Gap > 25%: significant overfitting — reduce complexity or add data

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, evals) in enumerate(xgb_evals.items()):
    if idx >= len(axes):
        break
    ax = axes[idx]

    # Find available metric
    metrics_available = list(evals.get("validation_0", {}).keys())
    metric_key = "mae" if "mae" in metrics_available else metrics_available[0] if metrics_available else None

    if metric_key and "validation_0" in evals and "validation_1" in evals:
        train_scores = evals["validation_0"][metric_key]
        val_scores   = evals["validation_1"][metric_key]

        ax.plot(train_scores, label=f"Train {metric_key}", color="steelblue", alpha=0.8)
        ax.plot(val_scores, label=f"Val {metric_key}", color="darkorange", alpha=0.8)

        # ── Overfitting diagnostic ────────────────────────────────────────────
        # Use last 10% of iterations as convergence region
        n_iters = len(val_scores)
        tail = max(1, n_iters // 10)
        train_converged = np.mean(train_scores[-tail:])
        val_converged   = np.mean(val_scores[-tail:])

        if train_converged > 0:
            gap_pct = abs(val_converged - train_converged) / train_converged * 100
        else:
            gap_pct = 0

        # Color-coded gap annotation
        if gap_pct < 10:
            gap_color, gap_label = "green", "Healthy"
        elif gap_pct < 25:
            gap_color, gap_label = "orange", "Mild overfit"
        else:
            gap_color, gap_label = "red", "Overfitting"

        ax.axhline(val_converged, color="darkorange", linestyle="--", alpha=0.3)
        ax.text(0.98, 0.95, f"Gap: {gap_pct:.1f}% ({gap_label})",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=10, color=gap_color,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

        ax.set_xlabel("Iteration")
        ax.set_ylabel(metric_key.upper())
        ax.legend()
    ax.set_title(name)

plt.suptitle("Training Curves with Overfitting Diagnostics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 17. Residual Analysis — Where Does the Model Fail?

**Why residuals matter:** Aggregate metrics hide systematic failure modes. The model might be
excellent on average but terrible during peak hours or in sparse zones.

**We analyze:**
1. **Residual distribution** — skew reveals systematic over/under-prediction
2. **Residuals by demand level** — does the model fail on peaks, zeros, or mid-range?
3. **Residuals by hour** — time-of-day bias detection
4. **Worst predictions** — individual failure cases for debugging

In [ ]:
best_test_pred = np.clip(best_xgb.predict(X_test[FEATURE_COLS]), 0, None)
residuals = y_test - best_test_pred

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── 1. Residual distribution ─────────────────────────────────────────────────
ax = axes[0, 0]
ax.hist(residuals, bins=100, color="steelblue", alpha=0.7, edgecolor="white")
ax.axvline(0, color="red", linestyle="--", linewidth=1.5)
ax.set_title(f"Residual Distribution (mean={residuals.mean():.3f}, std={residuals.std():.3f})")
ax.set_xlabel("Actual - Predicted")
ax.set_ylabel("Count")
skew = pd.Series(residuals).skew()
ax.text(0.98, 0.95, f"Skew: {skew:.2f}\n{'Under-predicting peaks' if skew > 0.5 else 'Balanced' if abs(skew) < 0.5 else 'Over-predicting'}",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))

# ── 2. Residuals by demand level ─────────────────────────────────────────────
ax = axes[0, 1]
demand_bins = pd.cut(y_test, bins=[0, 0.5, 2, 5, 10, 50, y_test.max() + 1],
                     labels=["0", "1-2", "3-5", "6-10", "11-50", "50+"], ordered=True)
residual_by_demand = pd.DataFrame({"residual": residuals, "demand_bin": demand_bins})
residual_by_demand.boxplot(column="residual", by="demand_bin", ax=ax)
ax.axhline(0, color="red", linestyle="--", alpha=0.5)
ax.set_title("Residuals by Demand Level")
ax.set_xlabel("Actual Demand Range")
ax.set_ylabel("Residual")
plt.sca(ax)
plt.title("Residuals by Demand Level")

# ── 3. Residuals by hour ─────────────────────────────────────────────────────
ax = axes[1, 0]
if "hour_sin" in X_test.columns:
    # Recover hour from sin/cos (approximate)
    hour_approx = np.round(np.arctan2(
        X_test["hour_sin"].values,
        X_test["hour_cos"].values
    ) * 24 / (2 * np.pi) % 24).astype(int)
elif "hour" in X_test.columns:
    hour_approx = X_test["hour"].values.astype(int)
else:
    hour_approx = np.zeros(len(residuals))

hour_mae = pd.DataFrame({"hour": hour_approx, "abs_error": np.abs(residuals)})
hourly_stats = hour_mae.groupby("hour")["abs_error"].agg(["mean", "std"])
ax.bar(hourly_stats.index, hourly_stats["mean"], color="steelblue", alpha=0.7)
ax.errorbar(hourly_stats.index, hourly_stats["mean"], yerr=hourly_stats["std"],
            fmt="none", color="black", alpha=0.3)
ax.set_title("MAE by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("Mean Absolute Error")

# ── 4. Worst predictions ─────────────────────────────────────────────────────
ax = axes[1, 1]
abs_errors = np.abs(residuals)
worst_idx = np.argsort(abs_errors)[-100:]
ax.scatter(y_test[worst_idx], best_test_pred[worst_idx], alpha=0.5, s=15, color="crimson")
max_val = max(y_test[worst_idx].max(), best_test_pred[worst_idx].max())
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.5)
ax.set_title("Worst 100 Predictions: Actual vs Predicted")
ax.set_xlabel("Actual Demand")
ax.set_ylabel("Predicted Demand")

plt.suptitle(f"Residual Analysis — {best_xgb_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Systematic failure summary ────────────────────────────────────────────────
print("\nFailure Mode Summary:")
zero_mask = y_test == 0
peak_mask = y_test > np.percentile(y_test, 95)
mid_mask  = ~zero_mask & ~peak_mask

print(f"  Zero-demand rows:  MAE={np.mean(np.abs(residuals[zero_mask])):.4f} "
      f"(mean pred={best_test_pred[zero_mask].mean():.3f}, should be ~0)")
print(f"  Mid-range rows:    MAE={np.mean(np.abs(residuals[mid_mask])):.4f}")
print(f"  Peak rows (>P95):  MAE={np.mean(np.abs(residuals[peak_mask])):.4f} "
      f"(mean actual={y_test[peak_mask].mean():.1f}, mean pred={best_test_pred[peak_mask].mean():.1f})")

## 18. Feature Importance — Beyond Built-in Gain

**Problem with built-in importance:** XGBoost's `feature_importances_` (gain-based) can overweight
high-cardinality or noisy features. It shows how much the model *uses* a feature, not how much
it *needs* it.

**Three perspectives:**
1. **Built-in (gain):** How much does each feature reduce the loss when used in splits?
2. **Permutation importance:** How much does test performance degrade when a feature's values
   are randomly shuffled? This measures actual predictive contribution.
3. **Grouped importance:** Sum permutation importance by feature group to answer the key question:
   "Do the new engineered features matter, or is the model still dominated by lags?"

**Why this answers the right question:** If grouped importance shows lag >> everything else,
the feature engineering added complexity without value. If non-lag groups contribute significantly,
the engineering effort is justified.

In [ ]:
# ── 1. Built-in Gain Importance (top 25) ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

importance_gain = pd.Series(
    best_xgb.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=True).tail(25)

axes[0].barh(importance_gain.index, importance_gain.values, color="steelblue")
axes[0].set_title(f"{best_xgb_name} — Top 25 Features (Gain)")
axes[0].set_xlabel("Importance (Gain)")

# ── 2. Permutation Importance (on val set for speed) ──────────────────────────
print("Computing permutation importance (this may take a minute)...")
perm_result = permutation_importance(
    best_xgb, X_val[FEATURE_COLS], y_val,
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1,
    scoring="neg_mean_absolute_error"
)

perm_imp = pd.Series(
    perm_result.importances_mean, index=FEATURE_COLS
).sort_values(ascending=True).tail(25)

axes[1].barh(perm_imp.index, perm_imp.values, color="darkorange")
axes[1].set_title(f"{best_xgb_name} — Top 25 Features (Permutation)")
axes[1].set_xlabel("MAE Increase When Shuffled")

plt.suptitle("Feature Importance: Built-in vs Permutation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ── 3. Grouped Importance ────────────────────────────────────────────────────
print("\n" + "="*60)
print("GROUPED FEATURE IMPORTANCE (Permutation-Based)")
print("="*60)

perm_all = pd.Series(perm_result.importances_mean, index=FEATURE_COLS)
group_importance = {}
for group_name, group_cols in FEATURE_GROUPS.items():
    cols_present = [c for c in group_cols if c in perm_all.index]
    if cols_present:
        group_importance[group_name] = perm_all[cols_present].sum()

group_imp_sorted = pd.Series(group_importance).sort_values(ascending=False)
total_imp = group_imp_sorted.sum()

for name, imp in group_imp_sorted.items():
    pct = imp / total_imp * 100 if total_imp > 0 else 0
    bar = "#" * int(pct / 2)
    print(f"  {name:12s}: {imp:.4f}  ({pct:5.1f}%)  {bar}")

# ── Key finding ───────────────────────────────────────────────────────────────
lag_pct = group_importance.get("lag", 0) / total_imp * 100 if total_imp > 0 else 0
non_lag_pct = 100 - lag_pct
print(f"\nKey Finding:")
print(f"  Lag features: {lag_pct:.1f}% of total importance")
print(f"  Non-lag features: {non_lag_pct:.1f}% of total importance")
if lag_pct > 70:
    print("  Conclusion: Model is still lag-dominated. Engineered features have marginal impact.")
elif lag_pct > 40:
    print("  Conclusion: Both lag and engineered features contribute. The feature engineering adds value.")
else:
    print("  Conclusion: Engineered features dominate. Strong value from feature engineering.")

# Plot grouped importance
fig, ax = plt.subplots(figsize=(10, 5))
group_imp_sorted.plot(kind="barh", ax=ax, color="teal")
ax.set_xlabel("Summed Permutation Importance")
ax.set_title("Feature Group Importance — Are Engineered Features Worth It?")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 19. Multi-AOI Visualization — Busy, Medium, and Sparse

**Problem with old notebook:** Only showed the busiest AOI, which cherry-picks the best-looking
result. A fair evaluation must show how the model performs across the demand spectrum.

**Approach:** Select one representative AOI from each busyness tier (top 10%, median, bottom 10%)
and plot actual vs predicted vs lag-24 baseline for each.

In [ ]:
# ── Select representative AOIs from each tier ─────────────────────────────────
test_demand = pd.DataFrame({
    "aoi_id": test_meta["aoi_id"].values,
    "actual": y_test,
    "pred": best_test_pred,
    "lag24": X_test["lag_24"].values if "lag_24" in X_test.columns else np.nan,
    "bucket_hour": test_meta["bucket_hour"].values,
})

aoi_total = test_demand.groupby("aoi_id")["actual"].sum().sort_values(ascending=False)
n_aois = len(aoi_total)

# Pick representative AOIs
busy_aoi   = aoi_total.index[max(0, int(n_aois * 0.05))]       # top 5%
medium_aoi = aoi_total.index[int(n_aois * 0.50)]                # median
sparse_aoi = aoi_total.index[min(n_aois - 1, int(n_aois * 0.90))]  # bottom 10%

tier_aois = [
    ("Busy AOI (top 5%)", busy_aoi),
    ("Medium AOI (median)", medium_aoi),
    ("Sparse AOI (bottom 10%)", sparse_aoi),
]

fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=False)

for idx, (tier_name, aoi_id) in enumerate(tier_aois):
    ax = axes[idx]
    aoi_data = test_demand[test_demand["aoi_id"] == aoi_id].sort_values("bucket_hour")

    if len(aoi_data) == 0:
        ax.text(0.5, 0.5, f"No test data for AOI {aoi_id}", transform=ax.transAxes, ha="center")
        continue

    ax.plot(aoi_data["bucket_hour"].values, aoi_data["actual"].values,
            label="Actual", color="black", linewidth=1.5, alpha=0.8)
    ax.plot(aoi_data["bucket_hour"].values, aoi_data["pred"].values,
            label=f"Predicted ({best_xgb_name})", color="steelblue", linewidth=1.2, alpha=0.8)
    if "lag_24" in X_test.columns:
        ax.plot(aoi_data["bucket_hour"].values, aoi_data["lag24"].values,
                label="Lag-24 Baseline", color="gray", linewidth=0.8, linestyle="--", alpha=0.6)

    aoi_mae = np.mean(np.abs(aoi_data["actual"].values - aoi_data["pred"].values))
    aoi_total_demand = aoi_data["actual"].sum()
    ax.set_title(f"{tier_name} — AOI {aoi_id} | MAE={aoi_mae:.2f} | Total demand={aoi_total_demand:.0f}")
    ax.set_ylabel("Demand Count")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time")
plt.suptitle("Model Performance Across AOI Busyness Tiers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Per-tier aggregate metrics ────────────────────────────────────────────────
busyness_test = X_test["aoi_busyness"].values if "aoi_busyness" in X_test.columns else None
if busyness_test is not None:
    tiers = pd.qcut(busyness_test, q=3, labels=["sparse", "medium", "busy"])
    print(f"\n{'Tier':<10} {'N rows':>8} {'MAE':>8} {'RMSE':>8} {'RMSLE':>8}")
    print("-" * 46)
    for tier in ["sparse", "medium", "busy"]:
        mask = tiers == tier
        m = compute_metrics(y_test[mask], best_test_pred[mask])
        print(f"{tier:<10} {mask.sum():>8,} {m['mae']:>8.4f} {m['rmse']:>8.4f} {m['rmsle']:>8.4f}")

## 20. Save Models & Final Summary

In [ ]:
# ── Save models ───────────────────────────────────────────────────────────────
os.makedirs(CONFIG["model_dir"], exist_ok=True)

# Save best XGBoost
best_xgb_path = os.path.join(CONFIG["model_dir"], f"{best_xgb_name.replace(' ', '_')}.json")
best_xgb.save_model(best_xgb_path)
print(f"Saved best XGBoost: {best_xgb_path}")

# Save LightGBM
lgb_path = os.path.join(CONFIG["model_dir"], "LightGBM_Poisson.txt")
lgb_model.booster_.save_model(lgb_path)
print(f"Saved LightGBM: {lgb_path}")

# Save all XGBoost variants
for name, mdl in xgb_models.items():
    path = os.path.join(CONFIG["model_dir"], f"{name.replace(' ', '_')}.json")
    mdl.save_model(path)
print(f"Saved {len(xgb_models)} XGBoost models to {CONFIG['model_dir']}")

# ── Final Summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("TRAINING PIPELINE COMPLETE")
print("=" * 70)

# Best model
best_overall = min(EXPERIMENT_LOG, key=lambda x: x.get("test_mae", float("inf")))
print(f"\nBest model: {best_overall['name']}")
print(f"  Test MAE:     {best_overall['test_mae']:.4f}")
print(f"  Test RMSE:    {best_overall['test_rmse']:.4f}")
print(f"  Test RMSLE:   {best_overall['test_rmsle']:.4f}")
print(f"  Test Poisson: {best_overall.get('test_poisson_dev', 'N/A')}")

# Improvement over baselines
if baseline_lag24_mae:
    imp = (1 - best_overall['test_mae'] / baseline_lag24_mae) * 100
    print(f"\nImprovement over Lag-24: {imp:+.1f}%")

# Walk-forward stability
if wf_results:
    print(f"\nWalk-Forward Stability: MAE std={wf_df['mae'].std():.4f} "
          f"(CV={cv_coeff:.1f}%)")

# Feature importance key finding
print(f"\nFeature Engineering Impact:")
print(f"  Lag group: {lag_pct:.1f}% of permutation importance")
print(f"  Non-lag groups: {non_lag_pct:.1f}% of permutation importance")

# Ablation key finding
if ablation_results:
    full_mae = ablation_results["Full"]["test"]["mae"]
    lag_mae  = ablation_results["Lag-Only"]["test"]["mae"]
    imp_abl = (1 - full_mae / lag_mae) * 100
    print(f"  Full features vs Lag-only: {imp_abl:+.1f}% MAE improvement")

print(f"\nAll {len(EXPERIMENT_LOG)} experiments saved to {CONFIG['experiment_log']}")
print("Models saved to:", CONFIG["model_dir"])